# HW5P4 — SQL practice

**Your name:** Mengyang Hu

Write five SQL queries (aggregation, joins, filtering) against a real dataset and explain each one. Complete each `TODO` section.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')

def q(sql, params=()):
    return pd.read_sql_query(sql, conn, params=params)

## Task 1: Load a real dataset

TODO: read your CSV file(s) and write them to SQLite. Then say what one row of each table represents.

In [46]:
# TODO: for example
a = pd.read_csv('Spotify most streamed.csv')
b = pd.read_csv('listeners.csv')

# transfer data from string to int
for col in ['Streams', 'Daily']:
    if col in a.columns:
        clean_s = a[col].astype(str).str.replace(',', '').str.strip()
        a[col] = pd.to_numeric(clean_s, errors='coerce').astype('Int64')

for col in ['Listeners', 'Daily Trend', 'PkListeners']:
    if col in b.columns:
        b[col] = b[col].astype(str).str.replace(',', '').astype(int)

# saperate artist and title into two columns
if 'Artist and Title' in a.columns:
    a[['Artist', 'Title']] = a['Artist and Title'].str.split('-', n=1, expand=True)


    a['Artist'] = a['Artist'].str.strip()
    a['Title'] = a['Title'].str.strip()


a.to_sql('sms', conn, index=False, if_exists='replace')
b.to_sql('listeners', conn, index=False, if_exists='replace')

2500

Row1 read csv file and save the dataset as variable 'a'.Row2 transfer a to sql file. Row3 and 4 do same thing as row 1 and 2 for second dataset. Each row in a represents an observation of artist's total and daily streams. Each row in b represents an observation of overall audience statistics, including artist, listeners, daily trend, peak, pklisteners.

## Task 2 and 3: Five queries

For each: a comment on its purpose, a row-count prediction, the query, the result.

In [47]:
# Query 1 (filter): TODO purpose: Find peak rank top 10 artists and pklisteners.
# Prediction: TODO rows: There are more than 10 artists in the results.
q('''SELECT *
  FROM listeners
  WHERE Peak<10;''')

,Artist,Listeners,Daily Trend,Peak,PkListeners
0,The Weeknd,107592328,-138880,1,113034886
1,Taylor Swift,101003302,889,2,101003302
2,Ed Sheeran,76475126,-68137,2,87934910
3,Dua Lipa,76421916,-71356,4,77778397
4,Bad Bunny,76162057,-199052,3,83950570
5,Rihanna,75784389,116405,2,80958750
6,Drake,75371611,-67013,6,76391086
7,Justin Bieber,72623228,34540,6,75467229
8,Billie Eilish,71793820,-72877,8,72368528
9,Miley Cyrus,71277599,71089,2,84140935


In [48]:
# Query 2 (aggregation): TODO purpose: Find the max of pklisteners and artist.
# Prediction: TODO rows: The weekend has the highest pklisteners.
q('''SELECT artist, MAX(pklisteners)
     FROM listeners
     LIMIT 1;''')

,Artist,MAX(pklisteners)
0,The Weeknd,113034886


In [53]:
# Query 4: TODO purpose: Find most promising artists.
# Prediction: TODO rows: There are some artists have daily trend higher than 100000.
q('''SELECT Artist, daily, "Daily Trend"
  FROM sms
  INNER JOIN listeners
  USING (artist)
  GROUP BY Artist
  ORDER BY "Daily Trend" DESC
  LIMIT 10;''')

,Artist,Daily,Daily Trend
0,Megan Thee Stallion,86759,743072
1,Olivia Rodrigo,1015488,732723
2,Doja Cat,830083,716207
3,Cardi B,368715,617636
4,Dove Cameron,466193,504512
5,Zach Bryan,1452268,418482
6,RAYE,891574,300641
7,The Rolling Stones,764893,277634
8,Bruno Mars,1292244,246177
9,Timbaland,667613,245021


In [58]:
# Query 5: TODO purpose:Find the difference between max and min streams songs.
# Prediction: TODO rows: The difference could be very big.
q('''SELECT MAX(Streams)-MIN(Streams) AS difference
  FROM sms;''')

,difference
0,3385673543


## Task 4: Check one result with pandas

TODO: reproduce one query with pandas and show the results agree.

In [63]:
# TODO TASK1
b[b['Peak']<10]


,Artist,Listeners,Daily Trend,Peak,PkListeners
0,The Weeknd,107592328,-138880,1,113034886
1,Taylor Swift,101003302,889,2,101003302
2,Ed Sheeran,76475126,-68137,2,87934910
3,Dua Lipa,76421916,-71356,4,77778397
4,Bad Bunny,76162057,-199052,3,83950570
5,Rihanna,75784389,116405,2,80958750
6,Drake,75371611,-67013,6,76391086
7,Justin Bieber,72623228,34540,6,75467229
8,Billie Eilish,71793820,-72877,8,72368528
9,Miley Cyrus,71277599,71089,2,84140935


## AI use disclosure

TODO: which AI tools (if any) you used and what you checked or changed. Write "None" if you did not use AI.

I used Gemini to transfer data from string into integer and debug. I checked the result with original data to see if the result is correct.